> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

# Building an Agent with LangChain v1 `create_agent`

We build a single agent that combines:

- A **Python `@tool`** (`search_wikipedia`)
- A **hosted Responses-API tool** (`web_search`)
- A `system_prompt`
- A streaming loop (`stream_mode="values"`) that surfaces intermediate tool calls

LangChain v1's `create_agent` is the successor to the deprecated `create_react_agent`; it returns a compiled LangGraph agent.

In [ ]:
%pip install langchain langchain-openai langgraph langsmith wikipedia

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

# Verify the import resolved (create_agent moved around in some v1 patch releases).
print(create_agent)

In [ ]:
import wikipedia

@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia and return a short summary for the given query."""
    try:
        return wikipedia.summary(query, sentences=3)
    except Exception as exc:  # disambiguation, no page, etc.
        return f"No clean Wikipedia result: {exc}"

### Hosted tool: `web_search`

The Responses API exposes hosted tools. We pass the `web_search` tool dict alongside our Python tool so the agent can choose live web results when Wikipedia is not enough.

In [ ]:
web_search_tool = {"type": "web_search"}

tools = [search_wikipedia, web_search_tool]

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise research assistant. Prefer search_wikipedia for stable, "
    "encyclopedic facts. Use web_search for recent or fast-changing information. "
    "Cite which tool you used."
)

agent = create_agent(
    "openai:gpt-5.5",
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Who wrote the novel Dune?"}]}
)
print(result["messages"][-1].content)

### Streaming with intermediate tool calls

`stream_mode="values"` yields the full state after each step, so we can watch the agent reason, call tools, and produce a final answer.

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What are the latest developments in fusion energy?"}]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

### LangSmith tracing

If `LANGCHAIN_TRACING_V2=true` (and `LANGCHAIN_API_KEY` is set), every `create_agent` run is traced automatically — no callback wiring needed. The cell below prints a link to your most recent traced run.

In [ ]:
import os
from langsmith import Client

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")

ls_client = Client()
project = os.environ.get("LANGCHAIN_PROJECT", "default")
runs = list(ls_client.list_runs(project_name=project, limit=1))
if runs:
    print("LangSmith trace URL:", ls_client.get_run_url(run=runs[0]))
else:
    print("No runs found yet — run the agent above with tracing enabled first.")